# Run an experiment

Pick the **data**, the **model**, and the **training recipe**, then train. Every run is saved automatically
(`runs/<site>/<run_name>/<timestamp>/`: trained model, config, metrics, predictions) and gets one row in
`results/experiment_log.csv`. Cells marked **CHOOSE** are the only ones you edit; run the notebook top to bottom.

| Step | What you choose | Where the code lives |
|---|---|---|
| 1 Data | site, history / future features, lookback, horizon, split | `pvf/data/` |
| 2 Model | architecture and its hyperparameters | `pvf/models/architectures/` |
| 3 Input adapter | (inspect) how the batch becomes this model's input | `pvf/model_inputs/` |
| 4 Recipe | loss, optimizer, scheduler, epochs, batch size | `pvf/training/` |
| 5-7 | train, diagnose, compare | `pvf/experiment.py`, `pvf/evaluation/` |

In [ ]:
%matplotlib inline
import pandas as pd
import pvf

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

## 0 Machine check (once per machine)
Python, GPU, packages, and that the data are identical to the committed checksums.

In [ ]:
pvf.check_environment()

## 1 Data
Available features. `usable_as_future_feature` = known in advance (clear sky, calendar, yesterday's power); `*_oracle` features are measured values used as a perfect forecast (an upper bound only).

In [ ]:
pvf.describe_features()

In [ ]:
# CHOOSE the data
data_config = pvf.DataConfig(
    site="ssp",                                   # "ssp" (Sirajganj, 2025) | "nist" (Gaithersburg, 2017)
    history_feature_names=["ac_power", "irradiance", "module_temperature", "clear_sky_irradiance",
                           "hour_of_day_sin", "hour_of_day_cos", "day_of_year_sin", "day_of_year_cos"],
    future_feature_names=["clear_sky_irradiance", "ac_power_yesterday",
                          "hour_of_day_sin", "hour_of_day_cos", "day_of_year_sin", "day_of_year_cos"],
    lookback_steps=128,                           # x 30 min of history
    horizon_steps=48,                             # x 30 min ahead (48 = 24 h)
    split=pvf.SplitConfig(mode="blocked"),        # weekly blocks: 4 train / 1 validation / 1 test, every season
)
data_bundle = pvf.build_data_bundle(data_config)
pvf.describe_data_bundle(data_bundle)

## 2 Model
Every registered model and its hyperparameters (constructor arguments) with defaults.

In [ ]:
pvf.describe_models()

In [ ]:
# CHOOSE the model: any model_name from the table above, and any of its hyperparameters
model_name = "patch_tst"
model_hyperparameters = {"model_size": 64, "n_layers": 2}
residual_on_smart_persistence = False             # True: forecast = smart persistence + learned correction

model = pvf.build_model(model_name, data_bundle.spec, residual_on_smart_persistence, **model_hyperparameters)
model

## 3 Input adapter
How the standard batch from the data pipeline becomes this model's input, step by step.

In [ ]:
batch = pvf.sample_batch(data_bundle)
print(model.input_adapter)
model.input_adapter.trace(batch)

In [ ]:
model.input_adapter.plot(batch);

## 4 Training recipe

In [ ]:
print("losses:", list(pvf.LOSSES), "| optimizers:", list(pvf.OPTIMIZERS), "| schedulers:", list(pvf.SCHEDULERS))

# CHOOSE the training recipe
recipe = pvf.TrainingRecipe(
    loss="mse",                     # quantile models always use "pinball"
    optimizer="adamw",
    learning_rate=1e-3,
    weight_decay=1e-4,
    scheduler="cosine",             # scheduler_options={...} for extra arguments
    n_epochs=50,
    batch_size=64,
    early_stopping_patience=10,
)
run_name = f"{model_name}_{data_config.site}"      # label used in comparisons

## 5 Train and score
Saves the trained model and all results; prints standard metrics per split.

In [ ]:
experiment_config = pvf.ExperimentConfig(model_name=model_name, model_hyperparameters=model_hyperparameters,
                                         residual_on_smart_persistence=residual_on_smart_persistence,
                                         data=data_config, recipe=recipe, run_name=run_name)
result = pvf.run_experiment(experiment_config, data_bundle=data_bundle, model=model)
result.summary()

## 6 Diagnose this run

In [ ]:
pvf.plot_training_history(result.run_dir);

In [ ]:
pvf.plot_forecast_examples(result.run_dir, split="test", n_examples=3);

In [ ]:
pvf.plot_per_horizon([result.run_dir], metric="nrmse_pct", split="test");

In [ ]:
pvf.plot_error_by_hour([result.run_dir], split="test");

## 7 Compare runs
Latest run per run name, best first. Column meanings: `pvf.evaluation.metrics.METRIC_DESCRIPTIONS`.

In [ ]:
comparison = pvf.compare_runs(site=data_config.site, split="test")
comparison

In [ ]:
best_runs = comparison.run_dir[:5]
pvf.plot_per_horizon(best_runs, metric="nrmse_pct", split="test");
pvf.plot_error_by_hour(best_runs, split="test");

## 8 Fusing models
`pvf/models/architectures/fusion_example.py` fuses iTransformer's variable encoder with HDCF-Mamba's temporal encoder
through a learned gate, and its docstring lists the 4-step recipe. To try it, set `model_name = "fusion_itransformer_hdcf"`
in section 2. To build your own: copy that file, swap in two other encoders, and register it under a new name.

## 9 Reload or repeat a saved run

In [ ]:
trained_model = pvf.load_trained_model(result.run_dir)        # the trained model, ready to predict
# pvf.rerun_experiment(result.run_dir)                          # repeat the run exactly (e.g. on the other PC)